# DBLP title lookup (streaming)

This notebook builds a lightweight SQLite index from the 4GB DBLP XML file using streaming parsing, then queries by title. It only parses titles (plus optional metadata) and avoids loading the full XML into memory.

- Input: `dblp.xml` (and `dblp.dtd` alongside)
- Output: `dblp_titles.sqlite` (indexed by normalized title)


In [1]:
import os
import re
import sqlite3
import unicodedata
from typing import Iterable, Optional

from lxml import etree as LET

In [6]:
DBLP_XML = r"d:\Project\GhostCite\Experiment\4_llm_generated_citations\dblp\dblp.xml"
DB_PATH = r"d:\Project\GhostCite\Experiment\4_llm_generated_citations\dblp\dblp_titles.sqlite"

ENTRY_TAGS = {
    "article",
    "inproceedings",
    "proceedings",
    "book",
    "incollection",
    "phdthesis",
    "mastersthesis",
    "www",
}


def normalize_title(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def iter_dblp_titles(xml_path: str) -> Iterable[tuple]:
    # lxml with DTD entity resolution to handle &uuml; etc.
    context = LET.iterparse(
        xml_path,
        events=("end",),
        load_dtd=True,
        resolve_entities=True,
        huge_tree=True,
    )
    _, root = next(context)
    for _, elem in context:
        tag = elem.tag
        if tag not in ENTRY_TAGS:
            continue
        title_elem = elem.find("title")
        if title_elem is None or not (title_elem.text or "").strip():
            elem.clear()
            continue
        title = " ".join(title_elem.itertext()).strip()
        if not title:
            elem.clear()
            continue
        key = elem.attrib.get("key", "")
        year = (elem.findtext("year") or "").strip()
        venue = (elem.findtext("journal") or elem.findtext("booktitle") or "").strip()
        authors = [a.text.strip() for a in elem.findall("author") if a.text]
        yield (key, tag, title, normalize_title(title), year, venue, "; ".join(authors))
        elem.clear()
        root.clear()


def build_sqlite_index(xml_path: str, db_path: str, batch_size: int = 2000) -> None:
    if os.path.exists(db_path):
        os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("PRAGMA journal_mode = WAL")
    cur.execute("PRAGMA synchronous = NORMAL")
    cur.execute("PRAGMA temp_store = MEMORY")

    cur.execute(
        """
        CREATE TABLE entries (
            id INTEGER PRIMARY KEY,
            dblp_key TEXT,
            type TEXT,
            title TEXT,
            title_norm TEXT,
            year TEXT,
            venue TEXT,
            authors TEXT
        )
        """
    )
    cur.execute("CREATE INDEX idx_title_norm ON entries(title_norm)")

    buffer = []
    for row in iter_dblp_titles(xml_path):
        buffer.append(row)
        if len(buffer) >= batch_size:
            cur.executemany(
                "INSERT INTO entries (dblp_key, type, title, title_norm, year, venue, authors) VALUES (?, ?, ?, ?, ?, ?, ?)",
                buffer,
            )
            conn.commit()
            buffer.clear()

    if buffer:
        cur.executemany(
            "INSERT INTO entries (dblp_key, type, title, title_norm, year, venue, authors) VALUES (?, ?, ?, ?, ?, ?, ?)",
            buffer,
        )
        conn.commit()

    conn.close()


def find_by_title(db_path: str, title: str, limit: int = 5) -> list[tuple]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    title_norm = normalize_title(title)
    cur.execute(
        "SELECT dblp_key, type, title, year, venue, authors FROM entries WHERE title_norm = ? LIMIT ?",
        (title_norm, limit),
    )
    rows = cur.fetchall()
    conn.close()
    return rows


# 1) Build index (first run will take a while for 4GB XML)
# build_sqlite_index(DBLP_XML, DB_PATH)

In [7]:
# 2) Query by exact title match (normalized)
results = find_by_title(DB_PATH, "Attention Is All You Need")
results

[('conf/nips/VaswaniSPUJGKP17',
  'inproceedings',
  'Attention is All you Need.',
  '2017',
  'NIPS',
  'Ashish Vaswani; Noam Shazeer; Niki Parmar; Jakob Uszkoreit; Llion Jones; Aidan N. Gomez; Lukasz Kaiser; Illia Polosukhin'),
 ('journals/corr/VaswaniSPUJGKP17',
  'article',
  'Attention Is All You Need.',
  '2017',
  'CoRR',
  'Ashish Vaswani; Noam Shazeer; Niki Parmar; Jakob Uszkoreit; Llion Jones; Aidan N. Gomez; Lukasz Kaiser; Illia Polosukhin')]

## Redis index (faster lookups, higher memory)

This variant loads title -> entry list into Redis for faster queries. You need a local Redis server running and the Python package `redis` installed.

- Pros: very fast lookup
- Cons: higher RAM usage (titles and payloads live in memory)


In [ ]:
import json
import redis

REDIS_URL = "redis://localhost:6379/0"
TITLE_KEY_PREFIX = "dblp:title:"


def build_redis_index(
    xml_path: str,
    redis_url: str,
    batch_size: int = 2000,
    flush_db: bool = False,
) -> None:
    r = redis.Redis.from_url(redis_url)
    if flush_db:
        r.flushdb()

    pipe = r.pipeline(transaction=False)
    buffered = 0

    for row in iter_dblp_titles(xml_path):
        dblp_key, entry_type, title, title_norm, year, venue, authors = row
        payload = {
            "dblp_key": dblp_key,
            "type": entry_type,
            "title": title,
            "year": year,
            "venue": venue,
            "authors": authors,
        }
        redis_key = f"{TITLE_KEY_PREFIX}{title_norm}"
        pipe.rpush(redis_key, json.dumps(payload, ensure_ascii=False))
        buffered += 1

        if buffered >= batch_size:
            pipe.execute()
            buffered = 0

    if buffered:
        pipe.execute()


def find_by_title_redis(redis_url: str, title: str, limit: int = 5) -> list[dict]:
    r = redis.Redis.from_url(redis_url)
    redis_key = f"{TITLE_KEY_PREFIX}{normalize_title(title)}"
    items = r.lrange(redis_key, 0, max(0, limit - 1))
    return [json.loads(item) for item in items]


# 1) Build index into Redis (requires local Redis server)
# build_redis_index(DBLP_XML, REDIS_URL, flush_db=True)

# 2) Query by exact title match (normalized)
# results = find_by_title_redis(REDIS_URL, "Attention Is All You Need")
# results
